# Práctica: pre-entrenar un nano-modelo con el Quijote

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosqueteroweb/IAconceptos/blob/main/notebooks/nanomodelo-quijote.ipynb)

Vamos a **entrenar desde cero** un modelo de lenguaje diminuto (unos 3 millones de parámetros) usando solo el texto de
*Don Quijote de la Mancha*. Es la misma receta que el pre-entrenamiento de un LLM gigante —tokenizar, predecir el
siguiente token, corregir el error millones de veces—, pero a escala de juguete.


> **Cómo usar este cuaderno.** No hace falta saber programar. Ejecuta las celdas **en orden**, de arriba abajo,
> con el botón ▶ que aparece a su izquierda (o con **Mayús + Intro**). Lee el texto que hay antes de cada celda:
> explica qué va a pasar. Si algo falla, usa *Entorno de ejecución → Reiniciar y ejecutar todo*.


**Antes de empezar:** en Colab, menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.
Con GPU tarda unos 3 minutos; sin GPU, unos 15.

Basado en el ejemplo [nanoGPT](https://github.com/karpathy/nanoGPT) de Andrej Karpathy. Material del curso
[IA de cero a modelo propio](https://mosqueteroweb.github.io/IAconceptos/) · MosqueteroWeb · CC BY-SA 4.0.

## Paso 1. Descargar el libro

Descargamos el texto completo del libro desde el [Proyecto Gutenberg](https://www.gutenberg.org/browse/languages/es)
(dominio público) y quitamos la licencia que Gutenberg añade al principio y al final. Si el servidor principal no
responde, el código prueba con dos copias espejo.

Por defecto es el **Quijote** (número 2000). Para usar otro libro, cambia `LIBRO` por su número en Gutenberg. Algunos
en español: 320 (*Lazarillo de Tormes*), 16625 (*Libro de Buen Amor*), 29506 (*El sombrero de tres picos*),
49836 (*Niebla*), 17340 (*Marianela*), 58221 (*La Odisea*, traducción), 24536 (*Los cuatro jinetes del Apocalipsis*).

In [ ]:
import urllib.request

LIBRO = 2000   # número del libro en el Proyecto Gutenberg (2000 = Don Quijote)

ruta = "/".join(str(LIBRO)[:-1]) + f"/{LIBRO}/{LIBRO}-0.txt"
FUENTES = [
    f"https://www.gutenberg.org/cache/epub/{LIBRO}/pg{LIBRO}.txt",
    f"https://gutenberg.pglaf.org/{ruta}",
    f"https://mirrors.xmission.com/gutenberg/{ruta}",
]
texto = None
for url in FUENTES:
    try:
        peticion = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        bruto = urllib.request.urlopen(peticion, timeout=30).read().decode("utf-8")
        if "START OF" in bruto:
            texto = bruto
            break
    except Exception as error:
        print("No se pudo usar", url, "→", error)

inicio = texto.index("START OF")
inicio = texto.index("\n", inicio) + 1
fin = next((texto.index(m) for m in ("*** END OF", "End of the Project Gutenberg", "End of Project Gutenberg") if m in texto), len(texto))
texto = texto[inicio:fin].strip()
print(f"Caracteres del libro: {len(texto):,}")
print(texto[:400])

## Paso 2. El tokenizador

Un modelo solo entiende números. Nuestro tokenizador será el más simple posible: **cada carácter es un token**.
El vocabulario es la lista de caracteres distintos que aparecen en el libro (letras, tildes, signos…), y a cada uno
le damos un número. Los LLM reales usan tokens de **sub-palabra** (trozos como «caball» + «ero»), pero la idea es la misma.

In [ ]:
caracteres = sorted(set(texto))
tamano_vocabulario = len(caracteres)
a_numero = {c: i for i, c in enumerate(caracteres)}
a_caracter = {i: c for i, c in enumerate(caracteres)}

def codificar(cadena):
    return [a_numero[c] for c in cadena if c in a_numero]  # los caracteres que no están en el libro se ignoran

def decodificar(numeros):
    return "".join(a_caracter[n] for n in numeros)

print("Vocabulario:", "".join(caracteres))
print("Tamaño del vocabulario:", tamano_vocabulario)
print("«DON QUIJOTE» en tokens:", codificar("DON QUIJOTE"))
print("Y de vuelta a texto:", decodificar(codificar("DON QUIJOTE")))

## Paso 3. Preparar los datos

Convertimos el libro entero en una larga lista de números. Guardamos el 90 % para **entrenar** y el 10 % final
para **validar**: son páginas que el modelo nunca verá durante el entrenamiento y que sirven para comprobar si
generaliza o solo memoriza.

La tarea del modelo será siempre la misma: dado un trozo de texto, **predecir el siguiente carácter**.

In [ ]:
import torch

torch.manual_seed(1605)  # año de la primera edición: así los resultados son repetibles
dispositivo = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Entrenaremos en:", dispositivo)

datos = torch.tensor(codificar(texto), dtype=torch.long)
corte = int(0.9 * len(datos))
datos_entreno, datos_validacion = datos[:corte], datos[corte:]

CONTEXTO = 128   # cuántos caracteres "mira hacia atrás" el modelo
LOTE = 32        # cuántos fragmentos procesa a la vez

def lote(parte):
    fuente = datos_entreno if parte == "entreno" else datos_validacion
    posiciones = torch.randint(len(fuente) - CONTEXTO, (LOTE,))
    x = torch.stack([fuente[p:p + CONTEXTO] for p in posiciones])
    y = torch.stack([fuente[p + 1:p + CONTEXTO + 1] for p in posiciones])  # el mismo texto, desplazado un carácter
    return x.to(dispositivo), y.to(dispositivo)

x, y = lote("entreno")
print("Entrada :", repr(decodificar(x[0][:40].tolist())))
print("Objetivo:", repr(decodificar(y[0][:40].tolist())))

## Paso 4. La arquitectura: un Transformer en miniatura

Esta celda define el modelo. No hace falta entenderla línea a línea: es un **Transformer** como el de los módulos 5 y 8,
con sus piezas clave:

- **Embeddings**: convierten cada token (y su posición) en un vector de números.
- **Atención**: cada carácter «mira» a los anteriores para decidir qué es relevante.
- **Bloques apilados**: repetimos atención + red neuronal 4 veces.
- **Cabeza de salida**: da una probabilidad para cada carácter del vocabulario.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

DIM = 256       # tamaño de los vectores (embeddings)
CABEZAS = 4     # cabezas de atención
CAPAS = 4       # bloques Transformer
DROPOUT = 0.2   # apaga neuronas al azar durante el entrenamiento para evitar memorizar

class Bloque(nn.Module):
    def __init__(self):
        super().__init__()
        self.norma1 = nn.LayerNorm(DIM)
        self.atencion = nn.MultiheadAttention(DIM, CABEZAS, dropout=DROPOUT, batch_first=True)
        self.norma2 = nn.LayerNorm(DIM)
        self.red = nn.Sequential(nn.Linear(DIM, 4 * DIM), nn.GELU(), nn.Linear(4 * DIM, DIM), nn.Dropout(DROPOUT))

    def forward(self, x):
        T = x.shape[1]
        mascara = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)  # no mirar el futuro
        h = self.norma1(x)
        x = x + self.atencion(h, h, h, attn_mask=mascara, need_weights=False)[0]
        return x + self.red(self.norma2(x))

class NanoModelo(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(tamano_vocabulario, DIM)
        self.posiciones = nn.Embedding(CONTEXTO, DIM)
        self.bloques = nn.Sequential(*[Bloque() for _ in range(CAPAS)])
        self.norma = nn.LayerNorm(DIM)
        self.salida = nn.Linear(DIM, tamano_vocabulario)

    def forward(self, idx, objetivos=None):
        T = idx.shape[1]
        x = self.tokens(idx) + self.posiciones(torch.arange(T, device=idx.device))
        logits = self.salida(self.norma(self.bloques(x)))
        perdida = None
        if objetivos is not None:
            perdida = F.cross_entropy(logits.view(-1, logits.size(-1)), objetivos.view(-1))
        return logits, perdida

    @torch.no_grad()
    def generar(self, idx, nuevos, temperatura=0.8):
        for _ in range(nuevos):
            logits, _ = self(idx[:, -CONTEXTO:])
            probabilidades = F.softmax(logits[:, -1, :] / temperatura, dim=-1)
            siguiente = torch.multinomial(probabilidades, num_samples=1)  # muestreo: no siempre el más probable
            idx = torch.cat([idx, siguiente], dim=1)
        return idx

modelo = NanoModelo().to(dispositivo)
print(f"Parámetros del modelo: {sum(p.numel() for p in modelo.parameters()) / 1e6:.2f} millones")

## Paso 5. Antes de entrenar: un modelo que no sabe nada

Sus parámetros son números al azar, así que genera caracteres sin sentido. Fíjate en el resultado para compararlo
después.

In [ ]:
def escribir(comienzo="DON QUIJOTE", caracteres=300, temperatura=0.8):
    modelo.eval()
    inicio = torch.tensor([codificar(comienzo)], dtype=torch.long, device=dispositivo)
    print(decodificar(modelo.generar(inicio, caracteres, temperatura)[0].tolist()))
    modelo.train()

escribir()

## Paso 6. El pre-entrenamiento

Este es el bucle que, a gran escala, cuesta millones de euros. En cada paso:

1. Tomamos 32 fragmentos del libro al azar.
2. El modelo predice el siguiente carácter en cada posición.
3. Medimos el error con la **pérdida** (*loss*): cuanto más baja, mejor predice.
4. La **retropropagación** ajusta un poquito los 3 millones de parámetros para reducir ese error.

Cada 500 pasos mostramos la pérdida con textos de entrenamiento y de validación. Si las dos bajan juntas, el modelo
está aprendiendo de verdad; si la de validación se queda atrás, empieza a memorizar.

In [ ]:
import time

PASOS = 5000
optimizador = torch.optim.AdamW(modelo.parameters(), lr=3e-4)
historial = []

@torch.no_grad()
def evaluar():
    modelo.eval()
    resultado = {}
    for parte in ("entreno", "validacion"):
        resultado[parte] = torch.stack([modelo(*lote(parte))[1] for _ in range(50)]).mean().item()
    modelo.train()
    return resultado

comienzo = time.time()
for paso in range(PASOS + 1):
    if paso % 500 == 0:
        p = evaluar()
        historial.append((paso, p["entreno"], p["validacion"]))
        print(f"Paso {paso:5d} · pérdida entreno {p['entreno']:.3f} · validación {p['validacion']:.3f} · {time.time() - comienzo:4.0f} s")
    if paso == PASOS:
        break
    x, y = lote("entreno")
    _, perdida = modelo(x, y)
    optimizador.zero_grad(set_to_none=True)
    perdida.backward()
    optimizador.step()

In [ ]:
import matplotlib.pyplot as plt

pasos, entreno, validacion = zip(*historial)
plt.figure(figsize=(7, 4))
plt.plot(pasos, entreno, marker="o", label="entrenamiento")
plt.plot(pasos, validacion, marker="o", label="validación")
plt.xlabel("pasos de entrenamiento"); plt.ylabel("pérdida (menos es mejor)")
plt.title("Así aprende el nano-modelo"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Paso 7. ¡A escribir como Cervantes (más o menos)!

Ahora el modelo ha aprendido la ortografía, las palabras frecuentes y hasta el ritmo de los diálogos del libro.
No esperes coherencia: con 3 millones de parámetros y un solo libro no puede entender la historia. Pero compáralo
con el Paso 5.

Prueba a cambiar el **comienzo** y la **temperatura**: baja (0,3) da textos repetitivos y «seguros»; alta (1,3) da
textos más locos. Es el mismo parámetro que ajustas en LM Studio o ChatGPT (módulo 12).

In [ ]:
escribir("DON QUIJOTE", temperatura=0.8)
print("\n" + "-" * 60 + "\n")
escribir("—Mire vuestra merced", temperatura=0.5)

## Lo que has hecho

**Has hecho un pre-entrenamiento completo**, con los mismos pasos que un modelo gigante. La diferencia es de escala:
datos (2 MB frente a decenas de billones de tokens), parámetros (3 millones frente a cientos de miles de millones) y
cómputo (minutos en una GPU frente a meses en miles).

Pero este modelo tiene una arquitectura casera y un tokenizador inventado por nosotros, y LM Studio u Ollama solo
cargan arquitecturas conocidas (Llama, Qwen, Gemma…) en formato GGUF. En la segunda parte lo solucionamos.

---

# Segunda parte: un nano-modelo que puedes abrir en LM Studio

Vamos a repetir el pre-entrenamiento con dos cambios:

1. **Arquitectura estándar:** la de **Llama** (la de Meta), en miniatura, usando la librería *transformers* de
   Hugging Face. Así podremos convertirla a GGUF.
2. **Tokenizador profesional:** en lugar de un token por carácter, usamos el de **SmolLM2** (un modelo pequeño de
   Hugging Face), con unos 49 000 trozos de palabra. Fíjate en cuántos tokens ocupa ahora el libro.

El modelo crece hasta unos 16 millones de parámetros, casi todos en la tabla de *embeddings* (49 000 tokens × 256
números). Tarda unos 5-10 minutos con GPU.

In [ ]:
from transformers import AutoTokenizer, LlamaConfig, LlamaForCausalLM

tokenizador2 = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
ids = torch.tensor(tokenizador2(texto)["input_ids"])
print(f"El libro ocupa {len(texto):,} caracteres y {len(ids):,} tokens ({len(texto) / len(ids):.1f} caracteres por token)")
print("Así trocea una frase:", tokenizador2.tokenize("En un lugar de la Mancha, de cuyo nombre no quiero acordarme"))

corte2 = int(0.9 * len(ids))
entreno2, validacion2 = ids[:corte2], ids[corte2:]
CONTEXTO2 = 128

def lote2(parte):
    fuente = entreno2 if parte == "entreno" else validacion2
    posiciones = torch.randint(len(fuente) - CONTEXTO2, (LOTE,))
    return torch.stack([fuente[p:p + CONTEXTO2] for p in posiciones]).to(dispositivo)

configuracion = LlamaConfig(vocab_size=len(tokenizador2), hidden_size=DIM, intermediate_size=704,
                            num_hidden_layers=CAPAS, num_attention_heads=CABEZAS, num_key_value_heads=CABEZAS,
                            max_position_embeddings=256, tie_word_embeddings=True,
                            bos_token_id=tokenizador2.bos_token_id, eos_token_id=tokenizador2.eos_token_id)
modelo2 = LlamaForCausalLM(configuracion).to(dispositivo)
print(f"Parámetros: {sum(p.numel() for p in modelo2.parameters()) / 1e6:.1f} millones")

## Entrenar el nano-Llama

El mismo bucle de antes. La librería ya sabe desplazar el texto un token para calcular la pérdida.

In [ ]:
PASOS2 = 2000
optimizador2 = torch.optim.AdamW(modelo2.parameters(), lr=1e-3)
comienzo = time.time()
for paso in range(PASOS2 + 1):
    if paso % 250 == 0:
        modelo2.eval()
        with torch.no_grad():
            p_ent = torch.stack([modelo2(input_ids=(x := lote2("entreno")), labels=x).loss for _ in range(20)]).mean()
            p_val = torch.stack([modelo2(input_ids=(x := lote2("validacion")), labels=x).loss for _ in range(20)]).mean()
        modelo2.train()
        print(f"Paso {paso:5d} · pérdida entreno {p_ent:.3f} · validación {p_val:.3f} · {time.time() - comienzo:4.0f} s")
    if paso == PASOS2:
        break
    x = lote2("entreno")
    perdida = modelo2(input_ids=x, labels=x).loss
    optimizador2.zero_grad(set_to_none=True)
    perdida.backward()
    optimizador2.step()

modelo2.eval()
inicio = tokenizador2("En un lugar de la Mancha", return_tensors="pt").input_ids.to(dispositivo)
salida = modelo2.generate(inicio, max_new_tokens=80, do_sample=True, temperature=0.8, top_k=50,
                          pad_token_id=tokenizador2.eos_token_id)
print(tokenizador2.decode(salida[0]))

## Convertir a GGUF

Guardamos el modelo en el formato de Hugging Face y lo convertimos a **GGUF** con el *script* de llama.cpp, igual que
en la práctica de cuantización. Le ponemos una «plantilla de chat» mínima: como el modelo no sabe conversar, lo que
escribas en LM Studio se le pasará tal cual y él lo continuará.

In [ ]:
import json, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

tokenizador2.chat_template = "{% for message in messages %}{{ message['content'] }}{% endfor %}"
modelo2.save_pretrained("nano-hf")
tokenizador2.save_pretrained("nano-hf")

etiqueta = json.load(urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=1"))[0]["tag_name"]
urllib.request.urlretrieve(f"https://github.com/ggml-org/llama.cpp/archive/refs/tags/{etiqueta}.zip", "llama-src.zip")
with zipfile.ZipFile("llama-src.zip") as z:
    z.extractall(".")
shutil.rmtree("llama-src", ignore_errors=True)
Path(f"llama.cpp-{etiqueta}").rename("llama-src")

ARCHIVO = f"nano-libro-{LIBRO}.gguf"
r = subprocess.run([sys.executable, "llama-src/convert_hf_to_gguf.py", "nano-hf", "--outfile", ARCHIVO, "--outtype", "f16"],
                   capture_output=True, text=True)
print(r.stderr[-300:])
print(f"{ARCHIVO}: {Path(ARCHIVO).stat().st_size / 1e6:.1f} MB")

## Llévatelo a LM Studio

1. **Descarga el archivo GGUF.** En Colab, la celda siguiente lo descarga; si no, usa el panel de archivos de la
   izquierda (tres puntos → Descargar). Si ejecutas el cuaderno en tu ordenador, ya lo tienes en la carpeta.
2. **Impórtalo.** En una terminal: `lms import -c ruta/al/nano-libro-2000.gguf` (en Mac está en `~/.lmstudio/bin/lms`).
   Aparecerá en **Library**.
3. **Cárgalo y escríbele el principio de una frase** del libro, por ejemplo «En un lugar de la Mancha». Ponle una
   temperatura de 0,8 y un límite de respuesta de 100 tokens. Lo continuará con su castellano de Cervantes… sin sentido.

Compara la velocidad (tok/s) con la de un modelo de verdad: este es diminuto.

In [ ]:
try:
    from google.colab import files
    files.download(ARCHIVO)
except ImportError:
    print("No estás en Colab: el archivo está en", Path(ARCHIVO).resolve())

**Para seguir:** cambia `CAPAS`, `DIM` o `PASOS` y observa cómo cambian el tiempo y la pérdida. O sustituye el
Quijote por otro libro de Gutenberg (variable `LIBRO`, al principio).